# PhoBERT ESG Actionability Classification

In [17]:
# Configuration
CONFIG = {
    "model": {
        "name": "vinai/phobert-large",
        "max_length": 256
    },
    "training": {
        "epochs": 5,
        "batch_size": 16,
        "learning_rate": 2.0e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "use_class_weights": True,
        "weight_method": "balanced"
    },
    "paths": {
        "train_data": "data/labels/action_hybrid_train_split.parquet",
        "val_data": "data/labels/action_hybrid_val_split.parquet",
        "gold_test": "data/labels/action_gold.parquet",
        "output_dir": "outputs/models/action_phobert_large",
        "inference_input": "data/corpus/esg_sentences.parquet",
        "inference_weak": "data/labels/action_weak.parquet",
        "inference_output": "data/corpus/esg_sentences_with_actionability.parquet"
    }
}


In [10]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from tqdm import tqdm
import argparse

MODEL_NAME = CONFIG["model"]["name"]
OUTPUT_DIR = Path(CONFIG["paths"]["output_dir"])
LABELS = ["Implemented", "Planning", "Indeterminate"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for i, label in enumerate(LABELS)}

CONTEXT_BLOCK_TYPES = {"paragraph", "bullet_like", "kpi_like"}

class ActionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 256, use_context: bool = True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_context = use_context

        self.texts = []
        for _, row in df.iterrows():
            if use_context and row.get("block_type", "") in CONTEXT_BLOCK_TYPES:
                parts = []
                if pd.notna(row.get("ctx_prev")) and row["ctx_prev"]:
                    parts.append(str(row["ctx_prev"]))
                parts.append(str(row["sentence"]))
                if pd.notna(row.get("ctx_next")) and row["ctx_next"]:
                    parts.append(str(row["ctx_next"]))
                text = " ".join(parts)
            else:
                text = str(row["sentence"])
            self.texts.append(text)

        label_col = "final_action" if "final_action" in df.columns else "gold_action"
        if label_col in df.columns:
            self.labels = [LABEL2ID[label] for label in df[label_col]]
        else:
            self.labels = None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weight = torch.tensor(self.class_weights, device=logits.device, dtype=logits.dtype)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, len(LABELS)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    per_class_f1 = f1_score(labels, predictions, average=None)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "f1_implemented": per_class_f1[0],
        "f1_planning": per_class_f1[1],
        "f1_indeterminate": per_class_f1[2],
    }


In [11]:
def train(config):
    epochs = config["training"]["epochs"]
    batch_size = config["training"]["batch_size"]
    lr = config["training"]["learning_rate"]
    max_len = config["model"]["max_length"]
    use_weights = config["training"]["use_class_weights"]

    print("Loading training data...")
    train_df = pd.read_parquet(config["paths"]["train_data"])
    val_df = pd.read_parquet(config["paths"]["val_data"])

    print(f"Train: {len(train_df)}, Val: {len(val_df)}")
    print(f"Train distribution:\n{train_df['final_action'].value_counts()}")

    class_weights = None
    if use_weights:
        labels_array = train_df["final_action"].map(LABEL2ID).values
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=np.array([0, 1, 2]),
            y=labels_array
        )
        print(f"\nClass weights: {dict(zip(LABELS, class_weights))}")

    print(f"\nLoading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    train_dataset = ActionDataset(train_df, tokenizer, max_len)
    val_dataset = ActionDataset(val_df, tokenizer, max_len)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        weight_decay=config["training"]["weight_decay"],
        warmup_ratio=config["training"]["warmup_ratio"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print("\nStarting training...")
    trainer.train()

    print("\n" + "="*60)
    print("VALIDATION RESULTS")
    print("="*60)
    val_results = trainer.evaluate()
    for k, v in val_results.items():
        print(f"  {k}: {v:.4f}")

    gold_path = Path(config["paths"]["gold_test"])
    if gold_path.exists():
        print("\n" + "="*60)
        print("GOLD SET EVALUATION")
        print("="*60)
        gold_df = pd.read_parquet(gold_path)
        if "gold_action" in gold_df.columns:
            gold_dataset = ActionDataset(gold_df, tokenizer, max_len)
            gold_predictions = trainer.predict(gold_dataset)
            gold_preds = np.argmax(gold_predictions.predictions, axis=-1)
            gold_labels = [LABEL2ID[l] for l in gold_df["gold_action"]]

            print(f"  Gold Accuracy: {accuracy_score(gold_labels, gold_preds):.4f}")
            print(f"  Gold Macro-F1: {f1_score(gold_labels, gold_preds, average='macro'):.4f}")
            print("\nClassification Report (Gold):")
            print(classification_report(gold_labels, gold_preds, target_names=LABELS))

    final_path = OUTPUT_DIR / "final"
    trainer.save_model(str(final_path))
    tokenizer.save_pretrained(str(final_path))
    print(f"\nModel saved: {final_path}")

    return tokenizer, model


In [12]:
def batch_inference(texts, tokenizer, model, batch_size=32, max_len=256, device="cuda"):
    all_preds = []
    all_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Inference"):
        batch_texts = texts[i:i + batch_size]

        encoding = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )
        encoding = {k: v.to(device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**encoding)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return all_preds, all_probs


def run_inference(config, tokenizer=None, model=None):
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Device: {device}")

    if tokenizer is None or model is None:
        model_path = Path(config["paths"]["output_dir"]) / "final"
        print(f"Loading model from {model_path}...")
        tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
        model = AutoModelForSequenceClassification.from_pretrained(model_path)

    model.to(device)
    model.eval()

    print("Loading sentences for inference...")
    df = pd.read_parquet(config["paths"]["inference_input"])
    print(f"Total sentences: {len(df)}")

    # Use ActionDataset logic for consistency
    dataset_for_inference = ActionDataset(df, tokenizer, config["model"]["max_length"], use_context=True)
    texts = dataset_for_inference.texts

    batch_size = config["training"]["batch_size"] * 2
    preds, probs = batch_inference(texts, tokenizer, model, batch_size, config["model"]["max_length"], device)

    df["action_pred_raw"] = [ID2LABEL[p] for p in preds]
    df["action_prob_raw"] = [max(p) for p in probs]
    df["action_probs"] = probs

    df["action_pred"] = df["action_pred_raw"]
    df["action_prob"] = df["action_prob_raw"]
    df["override_reason"] = ""

    weak_path = Path(config["paths"]["inference_weak"])
    if weak_path.exists():
        weak_df = pd.read_parquet(weak_path)
        print(f"Loaded weak labels for override: {len(weak_df)}")
        weak_subset = weak_df[["sent_id", "weak_action", "weak_conf"]].copy()
        df = df.merge(weak_subset, on="sent_id", how="left")

        # Example override: if weak label has very high confidence and model is uncertain
        mask_override = (
            (df["weak_conf"] >= 0.8) &
            (df["action_prob_raw"] < 0.6) &
            (df["override_reason"] == "")
        )
        df.loc[mask_override, "action_pred"] = df.loc[mask_override, "weak_action"]
        df.loc[mask_override, "action_prob"] = df.loc[mask_override, "weak_conf"]
        df.loc[mask_override, "override_reason"] = "weak_override"

        df = df.drop(columns=["weak_action", "weak_conf"])

    df["action_probs"] = df["action_probs"].apply(
        lambda p: {ID2LABEL[i]: round(v, 4) for i, v in enumerate(p)}
    )
    df = df.drop(columns=["action_pred_raw", "action_prob_raw"])

    output_path = Path(config["paths"]["inference_output"])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)

    print(f"\n=== INFERENCE COMPLETE ===")
    print(f"Output saved to: {output_path}")
    print(f"\nActionability Distribution:")
    print(df["action_pred"].value_counts())

    return df


In [13]:
# 1. Train the model
trained_tokenizer, trained_model = train(CONFIG)

Loading training data...
Train: 1281, Val: 566
Train distribution:
final_action
Implemented      427
Planning         427
Indeterminate    427
Name: count, dtype: int64

Class weights: {'Implemented': np.float64(1.0), 'Planning': np.float64(1.0), 'Indeterminate': np.float64(1.0)}

Loading model: vinai/phobert-large


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 Implemented,F1 Planning,F1 Indeterminate
1,1.101030,1.077045,0.448763,0.343246,0.052402,0.282132,0.695205
2,0.841428,0.881945,0.632509,0.613696,0.645012,0.519481,0.676596
3,0.671413,0.845088,0.650177,0.633116,0.691076,0.537190,0.671082
4,0.426661,0.772480,0.697880,0.677676,0.715262,0.589372,0.728395
5,0.311888,0.819546,0.706714,0.689746,0.723112,0.613208,0.732919


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


VALIDATION RESULTS


  eval_loss: 0.8195
  eval_accuracy: 0.7067
  eval_macro_f1: 0.6897
  eval_f1_implemented: 0.7231
  eval_f1_planning: 0.6132
  eval_f1_indeterminate: 0.7329
  eval_runtime: 7.4585
  eval_samples_per_second: 75.8870
  eval_steps_per_second: 2.4130
  epoch: 5.0000

GOLD SET EVALUATION
  Gold Accuracy: 0.8193
  Gold Macro-F1: 0.8169

Classification Report (Gold):
               precision    recall  f1-score   support

  Implemented       0.82      0.75      0.78       166
     Planning       0.82      0.95      0.88       166
Indeterminate       0.81      0.77      0.79       166

     accuracy                           0.82       498
    macro avg       0.82      0.82      0.82       498
 weighted avg       0.82      0.82      0.82       498



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved: outputs/models/action_phobert_large/final


In [6]:
# Config for HuggingFace Hub
HF_REPO_NAME = "esg-action"
HF_USERNAME = "huypham71"

# Model card info
MODEL_CARD_INFO = {
    "language": "vi",
    "license": "mit",
    "library_name": "transformers",
    "pipeline_tag": "text-classification",
    "tags": [
        "esg",
        "esg-washing",
        "actionability",
        "banking",
        "vietnamese",
        "nlp",
        "sustainability"
    ],
}

from huggingface_hub import login, HfApi, whoami

try:
    user_info = whoami()
    HF_USERNAME = user_info["name"]
    print(f"Already logged in as: {HF_USERNAME}")
except Exception:
    login()
    user_info = whoami()
    HF_USERNAME = user_info["name"]
    print(f"Logged in as: {HF_USERNAME}")

Already logged in as: huypham71


In [7]:
model_card_content = f"""
"""
# Save model card
model_card_path = OUTPUT_DIR / "README.md"
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(model_card_content)

# Push model to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"Pushing model to: https://huggingface.co/{repo_id}")

# Create repo if not exists and upload
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
    private=False,  # Set to True if you want private repo
)

# Upload all files in OUT_DIR
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=repo_id,
    repo_type="model",
    commit_message=f"Upload ESG Action",
)

Pushing model to: https://huggingface.co/huypham71/esg-action


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:10176: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-1005/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  ...eckpoint-201/optimizer.pt:   1%|          | 16.5MB / 2.95GB            

  ...eckpoint-603/optimizer.pt:   0%|          | 95.0kB / 2.95GB            

  ...nt-1005/model.safetensors:   0%|          |  558kB / 1.48GB            

  ...eckpoint-402/optimizer.pt:   0%|          | 99.6kB / 2.95GB            

CommitInfo(commit_url='https://huggingface.co/huypham71/esg-action/commit/1a161372fd0396b2abfdb6c3bd6308c446f939be', commit_message='Upload ESG Action', commit_description='', oid='1a161372fd0396b2abfdb6c3bd6308c446f939be', pr_url=None, repo_url=RepoUrl('https://huggingface.co/huypham71/esg-action', endpoint='https://huggingface.co', repo_type='model', repo_id='huypham71/esg-action'), pr_revision=None, pr_num=None)

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "huypham71/esg-action"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [16]:
# 2. Run inference on the entire corpus
inferred_df = run_inference(CONFIG, tokenizer, model)
inferred_df.head()

Device: cuda
Loading sentences for inference...
Total sentences: 44924


Inference: 100%|██████████| 1404/1404 [30:27<00:00,  1.30s/it]


Loaded weak labels for override: 21889

=== INFERENCE COMPLETE ===
Output saved to: data/corpus/esg_sentences_with_actionability.parquet

Actionability Distribution:
action_pred
Indeterminate    27683
Implemented      15655
Planning          1586
Name: count, dtype: int64


,doc_id,bank,year,section_id,block_id,sent_id,sent_idx_in_block,sentence,ctx_prev,ctx_next,block_type,section_title,topic_probs,topic_pred,topic_prob,override_reason,action_probs,action_pred,action_prob
0,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b36,agribank_2020_s36_0,0,Agribank triển khai hiệu quả 07 chính sách tín...,,"Trong năm 2020, Agribank 07 lần giảm lãi suất ...",kpi_like,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.0035, 'G': 0.0014, 'Non_ESG': 0.0026, ...",S_community,0.942485,,"{'Implemented': 0.942, 'Planning': 0.0033, 'In...",Implemented,0.942013
1,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b36,agribank_2020_s36_1,1,"Trong năm 2020, Agribank 07 lần giảm lãi suất ...",Agribank triển khai hiệu quả 07 chính sách tín...,,kpi_like,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.0035, 'G': 0.0014, 'Non_ESG': 0.0026, ...",S_community,0.942485,,"{'Implemented': 0.942, 'Planning': 0.0033, 'In...",Implemented,0.942013
2,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b37,agribank_2020_s37_0,0,"Bên cạnh hoạt động kinh doanh, Agribank là doa...",,Agribank luôn dành kinh phí đáng kể để ủng hộ ...,kpi_like,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.0007, 'G': 0.0009, 'Non_ESG': 0.0013, ...",S_community,0.994608,,"{'Implemented': 0.1798, 'Planning': 0.0033, 'I...",Indeterminate,0.816944
3,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b37,agribank_2020_s37_1,1,Agribank luôn dành kinh phí đáng kể để ủng hộ ...,"Bên cạnh hoạt động kinh doanh, Agribank là doa...","Chung tay bảo vệ môi trường, ""Vì tương lai xan...",kpi_like,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.1682, 'G': 0.0028, 'Non_ESG': 0.0045, ...",S_community,0.817272,,"{'Implemented': 0.9171, 'Planning': 0.0041, 'I...",Implemented,0.917067
4,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b37,agribank_2020_s37_2,2,"Chung tay bảo vệ môi trường, ""Vì tương lai xan...",Agribank luôn dành kinh phí đáng kể để ủng hộ ...,,kpi_like,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.9891, 'G': 0.0011, 'Non_ESG': 0.0011, ...",E,0.989064,,"{'Implemented': 0.9828, 'Planning': 0.0038, 'I...",Implemented,0.982831
